# Chain Rule and Backpropagation - Examples

Practical demonstrations of backpropagation and the chain rule for neural networks.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Example 1: Scalar Chain Rule

For y = f(g(x)), the chain rule states:

$$\frac{dy}{dx} = \frac{dy}{du} \cdot \frac{du}{dx}$$

where u = g(x).

In [ ]:
print("y = sin(x²)")
print("Let u = x², then y = sin(u)")
print("\ndy/dx = dy/du · du/dx")
print("      = cos(u) · 2x")
print("      = 2x · cos(x²)")

In [ ]:
x = 1.0

# Analytical derivative
dy_dx_analytical = 2 * x * np.cos(x**2)

# Numerical derivative
h = 1e-7
y_plus = np.sin((x + h)**2)
y_minus = np.sin((x - h)**2)
dy_dx_numerical = (y_plus - y_minus) / (2 * h)

print(f"At x = {x}:")
print(f"  Analytical: dy/dx = {dy_dx_analytical:.6f}")
print(f"  Numerical:  dy/dx = {dy_dx_numerical:.6f}")

## Example 2: Vector Chain Rule

For a scalar loss L of a vector function y(x), the chain rule becomes:

$$\nabla_{\mathbf{x}} L = \mathbf{J}_{\mathbf{y}}^T \nabla_{\mathbf{y}} L$$

This is the foundation of backpropagation!

In [ ]:
print("L(y(x)) where:")
print("  x ∈ ℝ²")
print("  y = [x₁², x₁x₂]ᵀ ∈ ℝ²")
print("  L = y₁ + y₂ ∈ ℝ")

print("\n∂L/∂y = [1, 1]ᵀ")
print("\nJacobian of y:")
print("J_y = [2x₁  0  ]")
print("      [x₂   x₁ ]")

print("\n∂L/∂x = J_yᵀ · ∂L/∂y")
print("      = [2x₁  x₂] [1]   [2x₁ + x₂]")
print("        [0    x₁] [1] = [x₁      ]")

In [ ]:
x = np.array([2.0, 3.0])

def y_func(x):
    return np.array([x[0]**2, x[0]*x[1]])

def L_func(x):
    y = y_func(x)
    return y[0] + y[1]

# Analytical gradient
grad_analytical = np.array([2*x[0] + x[1], x[0]])

# Numerical gradient
grad_numerical = np.zeros(2)
h = 1e-7
for i in range(2):
    x_plus = x.copy(); x_plus[i] += h
    x_minus = x.copy(); x_minus[i] -= h
    grad_numerical[i] = (L_func(x_plus) - L_func(x_minus)) / (2*h)

print(f"At x = {x}:")
print(f"  Analytical: ∂L/∂x = {grad_analytical}")
print(f"  Numerical:  ∂L/∂x = {grad_numerical.round(6)}")

## Example 3: Simple Backpropagation

Network: x → Linear(w,b) → Sigmoid → MSE Loss
- z = wx + b
- y = σ(z)
- L = (y - t)²

In [ ]:
# Parameters
w = 0.5
b = 0.1
x = 2.0
t = 0.8  # Target

# Forward pass
z = w * x + b
y = 1 / (1 + np.exp(-z))  # sigmoid
L = (y - t)**2

print("--- Forward Pass ---")
print(f"  x = {x}, w = {w}, b = {b}")
print(f"  z = wx + b = {z}")
print(f"  y = σ(z) = {y:.4f}")
print(f"  L = (y - t)² = {L:.4f}")

In [ ]:
# Backward pass
dL_dy = 2 * (y - t)
dy_dz = y * (1 - y)  # sigmoid derivative
dz_dw = x
dz_db = 1

# Chain rule
dL_dz = dL_dy * dy_dz
dL_dw = dL_dz * dz_dw
dL_db = dL_dz * dz_db

print("--- Backward Pass ---")
print(f"  ∂L/∂y = 2(y - t) = {dL_dy:.4f}")
print(f"  ∂y/∂z = y(1-y) = {dy_dz:.4f}")
print(f"  ∂L/∂z = ∂L/∂y · ∂y/∂z = {dL_dz:.4f}")
print(f"  ∂L/∂w = ∂L/∂z · x = {dL_dw:.4f}")
print(f"  ∂L/∂b = ∂L/∂z = {dL_db:.4f}")

In [ ]:
# Numerical verification
h = 1e-7
def compute_loss(w, b):
    z = w * x + b
    y = 1 / (1 + np.exp(-z))
    return (y - t)**2

dL_dw_num = (compute_loss(w + h, b) - compute_loss(w - h, b)) / (2*h)
dL_db_num = (compute_loss(w, b + h) - compute_loss(w, b - h)) / (2*h)

print("--- Numerical Verification ---")
print(f"  ∂L/∂w: analytical = {dL_dw:.6f}, numerical = {dL_dw_num:.6f}")
print(f"  ∂L/∂b: analytical = {dL_db:.6f}, numerical = {dL_db_num:.6f}")

## Example 4: Two-Layer Network Backpropagation

Network: x → [W₁,b₁] → ReLU → [W₂,b₂] → MSE Loss

In [ ]:
np.random.seed(42)

# Dimensions
n_input = 3
n_hidden = 4
n_output = 2

# Initialize
W1 = np.random.randn(n_hidden, n_input) * 0.1
b1 = np.zeros(n_hidden)
W2 = np.random.randn(n_output, n_hidden) * 0.1
b2 = np.zeros(n_output)

x = np.random.randn(n_input)
t = np.random.randn(n_output)  # target

def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)

In [ ]:
# Forward pass
z1 = W1 @ x + b1
a1 = relu(z1)
z2 = W2 @ a1 + b2
L = np.sum((z2 - t)**2)

print("--- Forward Pass ---")
print(f"  x shape: {x.shape}")
print(f"  z1 shape: {z1.shape}, a1 shape: {a1.shape}")
print(f"  z2 shape: {z2.shape}")
print(f"  Loss L = {L:.4f}")

In [ ]:
# Backward pass
dL_dz2 = 2 * (z2 - t)                    # (n_output,)
dL_dW2 = np.outer(dL_dz2, a1)            # (n_output, n_hidden)
dL_db2 = dL_dz2                          # (n_output,)

dL_da1 = W2.T @ dL_dz2                   # (n_hidden,)
dL_dz1 = dL_da1 * relu_derivative(z1)    # (n_hidden,)
dL_dW1 = np.outer(dL_dz1, x)             # (n_hidden, n_input)
dL_db1 = dL_dz1                          # (n_hidden,)

print("--- Backward Pass ---")
print(f"  ∂L/∂z₂ shape: {dL_dz2.shape}")
print(f"  ∂L/∂W₂ shape: {dL_dW2.shape}")
print(f"  ∂L/∂a₁ shape: {dL_da1.shape}")
print(f"  ∂L/∂z₁ shape: {dL_dz1.shape}")
print(f"  ∂L/∂W₁ shape: {dL_dW1.shape}")

In [ ]:
# Numerical gradient check for W1
h = 1e-5
dL_dW1_numerical = np.zeros_like(W1)

for i in range(W1.shape[0]):
    for j in range(W1.shape[1]):
        W1_plus = W1.copy(); W1_plus[i, j] += h
        z1_plus = W1_plus @ x + b1
        a1_plus = relu(z1_plus)
        z2_plus = W2 @ a1_plus + b2
        L_plus = np.sum((z2_plus - t)**2)
        
        W1_minus = W1.copy(); W1_minus[i, j] -= h
        z1_minus = W1_minus @ x + b1
        a1_minus = relu(z1_minus)
        z2_minus = W2 @ a1_minus + b2
        L_minus = np.sum((z2_minus - t)**2)
        
        dL_dW1_numerical[i, j] = (L_plus - L_minus) / (2 * h)

rel_error = np.linalg.norm(dL_dW1 - dL_dW1_numerical) / (
    np.linalg.norm(dL_dW1) + np.linalg.norm(dL_dW1_numerical) + 1e-10
)

print("--- Gradient Check ---")
print(f"  Relative error for W₁: {rel_error:.2e}")
print(f"  Check {'PASSED ✓' if rel_error < 1e-5 else 'FAILED ✗'}")

## Example 5: Softmax + Cross-Entropy Backprop

The combined gradient of softmax + cross-entropy is elegantly simple:

$$\frac{\partial L}{\partial z} = p - y$$

where p = softmax(z) and y is the one-hot target.

In [ ]:
def softmax(z):
    exp_z = np.exp(z - np.max(z))
    return exp_z / np.sum(exp_z)

z = np.array([1.0, 2.0, 3.0])
y = np.array([0, 0, 1])  # One-hot label (class 2)

# Forward
p = softmax(z)
L = -np.sum(y * np.log(p + 1e-10))

print("Forward:")
print(f"  z = {z}")
print(f"  p = softmax(z) = {p.round(4)}")
print(f"  y = {y}")
print(f"  L = -Σ yᵢ log(pᵢ) = {L:.4f}")

In [ ]:
# Simple combined gradient
dL_dz_simple = p - y

print("Combined gradient (derived analytically):")
print(f"  ∂L/∂z = p - y = {dL_dz_simple.round(4)}")

In [ ]:
# Numerical verification
h = 1e-7
dL_dz_numerical = np.zeros(3)
for j in range(3):
    z_plus = z.copy(); z_plus[j] += h
    p_plus = softmax(z_plus)
    L_plus = -np.sum(y * np.log(p_plus + 1e-10))
    
    z_minus = z.copy(); z_minus[j] -= h
    p_minus = softmax(z_minus)
    L_minus = -np.sum(y * np.log(p_minus + 1e-10))
    
    dL_dz_numerical[j] = (L_plus - L_minus) / (2*h)

print(f"Numerical: {dL_dz_numerical.round(4)}")
print(f"Match: {np.allclose(dL_dz_simple, dL_dz_numerical)}")

## Example 6: Gradient Checking

Gradient checking compares analytical gradients with numerical gradients to verify correctness:

$$\frac{\partial L}{\partial \theta_i} \approx \frac{L(\theta_i + \epsilon) - L(\theta_i - \epsilon)}{2\epsilon}$$

In [ ]:
np.random.seed(42)

def f(theta):
    """Example function."""
    x, y, z = theta
    return x**2 * y + np.sin(z) * y

def grad_f(theta):
    """Analytical gradient."""
    x, y, z = theta
    return np.array([
        2*x*y,            # ∂f/∂x
        x**2 + np.sin(z), # ∂f/∂y
        np.cos(z) * y     # ∂f/∂z
    ])

def numerical_gradient(f, theta, h=1e-7):
    """Compute numerical gradient."""
    grad = np.zeros_like(theta)
    for i in range(len(theta)):
        theta_plus = theta.copy(); theta_plus[i] += h
        theta_minus = theta.copy(); theta_minus[i] -= h
        grad[i] = (f(theta_plus) - f(theta_minus)) / (2*h)
    return grad

In [ ]:
theta = np.array([1.0, 2.0, 3.0])

g_analytical = grad_f(theta)
g_numerical = numerical_gradient(f, theta)

print(f"θ = {theta}")
print(f"\nAnalytical gradient: {g_analytical.round(6)}")
print(f"Numerical gradient:  {g_numerical.round(6)}")

# Relative error
rel_error = np.linalg.norm(g_analytical - g_numerical) / (
    np.linalg.norm(g_analytical) + np.linalg.norm(g_numerical) + 1e-10
)

print(f"\nRelative error: {rel_error:.2e}")
print(f"Check {'PASSED ✓' if rel_error < 1e-5 else 'FAILED ✗'}")

In [ ]:
# Element-wise comparison
print("\nElement-wise comparison:")
for i, (a, n) in enumerate(zip(g_analytical, g_numerical)):
    rel = abs(a - n) / (abs(a) + abs(n) + 1e-10)
    status = '✓' if rel < 1e-5 else '✗'
    print(f"  θ[{i}]: analytical={a:.6f}, numerical={n:.6f}, rel_err={rel:.2e} {status}")

## Example 7: Batched Gradient Computation

For a batch of inputs X (batch_size × features), the backward pass involves matrix operations.

In [ ]:
np.random.seed(42)

batch_size = 4
n_features = 3
n_outputs = 2

X = np.random.randn(batch_size, n_features)
W = np.random.randn(n_features, n_outputs) * 0.1
b = np.zeros(n_outputs)
Y_true = np.random.randn(batch_size, n_outputs)

print(f"X shape: {X.shape}")
print(f"W shape: {W.shape}")

In [ ]:
# Forward pass
Z = X @ W + b
L = np.mean((Z - Y_true)**2)

print(f"Z shape: {Z.shape}")
print(f"Loss: {L:.4f}")

In [ ]:
# Backward pass
dL_dZ = 2 * (Z - Y_true) / batch_size  # (batch, output)
dL_dW = X.T @ dL_dZ                     # (features, output)
dL_db = np.sum(dL_dZ, axis=0)           # (output,)

print(f"∂L/∂Z shape: {dL_dZ.shape}")
print(f"∂L/∂W shape: {dL_dW.shape}")
print(f"∂L/∂b shape: {dL_db.shape}")

In [ ]:
# Verify gradient for W
h = 1e-5
dL_dW_numerical = np.zeros_like(W)

for i in range(W.shape[0]):
    for j in range(W.shape[1]):
        W_plus = W.copy(); W_plus[i, j] += h
        L_plus = np.mean((X @ W_plus + b - Y_true)**2)
        
        W_minus = W.copy(); W_minus[i, j] -= h
        L_minus = np.mean((X @ W_minus + b - Y_true)**2)
        
        dL_dW_numerical[i, j] = (L_plus - L_minus) / (2*h)

rel_error = np.linalg.norm(dL_dW - dL_dW_numerical) / (
    np.linalg.norm(dL_dW) + np.linalg.norm(dL_dW_numerical) + 1e-10
)
print(f"Gradient check for W: rel_error = {rel_error:.2e}")

## Example 8: Computational Graph

Visualizing how gradients flow through a computational graph.

In [ ]:
print("Expression: L = (a * b + c)²")
print("")
print("Computational graph:")
print("  a ───┐")
print("       ├──→ [×] ──→ d ─┐")
print("  b ───┘               │")
print("                       ├──→ [+] ──→ e ──→ [²] ──→ L")
print("  c ───────────────────┘")

In [ ]:
a, b, c = 2.0, 3.0, 1.0

# Forward pass
d = a * b      # multiplication
e = d + c      # addition
L = e ** 2     # square

print(f"Forward pass:")
print(f"  a = {a}, b = {b}, c = {c}")
print(f"  d = a × b = {d}")
print(f"  e = d + c = {e}")
print(f"  L = e² = {L}")

In [ ]:
# Backward pass
dL_de = 2 * e           # ∂L/∂e = 2e
de_dd = 1               # ∂e/∂d = 1
de_dc = 1               # ∂e/∂c = 1
dd_da = b               # ∂d/∂a = b
dd_db = a               # ∂d/∂b = a

dL_dd = dL_de * de_dd   # Chain rule
dL_dc = dL_de * de_dc
dL_da = dL_dd * dd_da
dL_db = dL_dd * dd_db

print(f"Backward pass:")
print(f"  ∂L/∂e = 2e = {dL_de}")
print(f"  ∂L/∂d = ∂L/∂e × 1 = {dL_dd}")
print(f"  ∂L/∂c = ∂L/∂e × 1 = {dL_dc}")
print(f"  ∂L/∂a = ∂L/∂d × b = {dL_da}")
print(f"  ∂L/∂b = ∂L/∂d × a = {dL_db}")

In [ ]:
# Verify using direct differentiation
print("\nVerification (analytical):")
print(f"  L = (ab + c)²")
print(f"  ∂L/∂a = 2(ab + c) × b = 2 × {e} × {b} = {2*e*b}")
print(f"  ∂L/∂b = 2(ab + c) × a = 2 × {e} × {a} = {2*e*a}")
print(f"  ∂L/∂c = 2(ab + c) × 1 = 2 × {e} = {2*e}")

## Example 9: Vanishing Gradient Problem

Deep networks with sigmoid activations suffer from vanishing gradients because σ'(z) ≤ 0.25.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

print("Deep network with sigmoid activations:")
print("x → σ → σ → σ → σ → ... → σ → L")
print("\nσ'(z) = σ(z)(1-σ(z)) ≤ 0.25")
print("After n layers: gradient × (0.25)ⁿ")

In [ ]:
n_layers = 10
z = 0.5  # Typical value

sigma_prime = sigmoid(z) * (1 - sigmoid(z))
print(f"σ'({z}) = {sigma_prime:.4f}")

gradient_factors = [sigma_prime ** n for n in range(1, n_layers + 1)]

print(f"\nGradient factors through layers:")
for n, factor in enumerate(gradient_factors, 1):
    print(f"  Layer {n:2d}: {factor:.2e}")

In [ ]:
# Visualize
layers = list(range(1, 21))
gradient_factor = [sigma_prime ** n for n in layers]

plt.figure(figsize=(10, 5))
plt.semilogy(layers, gradient_factor, 'b-o')
plt.xlabel('Number of Layers')
plt.ylabel('Gradient Factor (log scale)')
plt.title('Vanishing Gradient with Sigmoid Activation')
plt.grid(True)
plt.tight_layout()
plt.show()

print("\n--- ReLU solves this ---")
print("ReLU'(z) = 1 for z > 0")
print("Gradient doesn't shrink through layers!")

## Example 10: Layer Class Implementation

Implementing neural network layers as classes with forward/backward methods.

In [ ]:
class Linear:
    """Linear layer: y = Wx + b"""
    def __init__(self, in_features, out_features):
        self.W = np.random.randn(out_features, in_features) * 0.1
        self.b = np.zeros(out_features)
        self.grad_W = None
        self.grad_b = None
    
    def forward(self, x):
        self.x = x  # Cache for backward
        return self.W @ x + self.b
    
    def backward(self, grad_output):
        self.grad_W = np.outer(grad_output, self.x)
        self.grad_b = grad_output
        return self.W.T @ grad_output


class Sigmoid:
    """Sigmoid activation."""
    def forward(self, z):
        self.out = 1 / (1 + np.exp(-z))
        return self.out
    
    def backward(self, grad_output):
        return grad_output * self.out * (1 - self.out)


class MSELoss:
    """Mean squared error loss."""
    def forward(self, pred, target):
        self.pred = pred
        self.target = target
        return np.sum((pred - target)**2)
    
    def backward(self):
        return 2 * (self.pred - self.target)

In [ ]:
# Build network
np.random.seed(42)
linear1 = Linear(3, 4)
sigmoid = Sigmoid()
linear2 = Linear(4, 2)
loss_fn = MSELoss()

# Data
x = np.array([1.0, 2.0, 3.0])
target = np.array([0.5, 0.5])

In [ ]:
# Forward pass
z1 = linear1.forward(x)
a1 = sigmoid.forward(z1)
z2 = linear2.forward(a1)
loss = loss_fn.forward(z2, target)

print(f"Forward pass:")
print(f"  x → z1 → a1 → z2 → Loss")
print(f"  {x.shape} → {z1.shape} → {a1.shape} → {z2.shape} → scalar")
print(f"  Loss = {loss:.4f}")

In [ ]:
# Backward pass
grad_z2 = loss_fn.backward()
grad_a1 = linear2.backward(grad_z2)
grad_z1 = sigmoid.backward(grad_a1)
grad_x = linear1.backward(grad_z1)

print(f"Backward pass:")
print(f"  ∂L/∂z2 = {grad_z2.round(4)}")
print(f"  ∂L/∂a1 = {grad_a1.round(4)}")
print(f"  ∂L/∂z1 = {grad_z1.round(4)}")
print(f"  ∂L/∂x  = {grad_x.round(4)}")
print(f"\n  ∂L/∂W2 shape: {linear2.grad_W.shape}")
print(f"  ∂L/∂W1 shape: {linear1.grad_W.shape}")

## Summary

### Chain Rule Forms

| Form | Formula |
|------|--------|
| Scalar | dL/dx = dL/dy · dy/dx |
| Vector | ∇_x L = J_yᵀ ∇_y L |
| Matrix | J_{f∘g} = J_f · J_g |

### Backprop Rules

| Operation | Backward Rule |
|-----------|---------------|
| z = Wx + b | ∂L/∂x = Wᵀ(∂L/∂z), ∂L/∂W = (∂L/∂z)xᵀ |
| a = σ(z) | ∂L/∂z = (∂L/∂a) ⊙ σ'(z) |
| y = x₁ + x₂ | ∂L/∂x₁ = ∂L/∂x₂ = ∂L/∂y |
| y = Σ xᵢ | ∂L/∂xᵢ = ∂L/∂y |

### Key Insights
1. Backprop = chain rule applied in reverse order
2. Softmax + cross-entropy gradient: p - y
3. Gradient checking: relative error should be < 1e-5
4. ReLU prevents vanishing gradients